In [1]:
import os
import re
import sys
import nltk
import argparse
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional
from pycocoevalcap.spice.spice import SPICE
from nltk.translate.meteor_score import meteor_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

ModuleNotFoundError: No module named 'pycocoevalcap'

In [ ]:
_punct_re = re.compile(r"[^\wáéíóúüñçàèìòùäëïöüÁÉÍÓÚÜÑÇÀÈÌÒÙÄËÏÖÜ-]+", re.UNICODE)

In [ ]:
def normalize(text: str) -> str:
    """minúsculas + colapsar espacios + quitar puntuación “dura” (mantiene acentos y guiones)"""
    if not isinstance(text, str):
        return ""
    text = text.strip().lower()
    text = _punct_re.sub(" ", text)
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
def tokenize(text: str) -> List[str]:
    return normalize(text).split()

# -----------------------
# ROUGE-L (LCS)
# -----------------------
def lcs_len(x: List[str], y: List[str]) -> int:
    m, n = len(x), len(y)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m):
        xi = x[i]
        for j in range(n):
            if xi == y[j]:
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])
    return dp[m][n]

In [ ]:
def rouge_l(reference: List[str], hypothesis: List[str], beta: float = 1.2) -> float:
    """
    ROUGE-L F-score (como en trabajos de captioning).
    beta=1.2 es común para dar algo más de peso al recall.
    """
    if len(reference) == 0 or len(hypothesis) == 0:
        return 0.0
    lcs = lcs_len(reference, hypothesis)
    prec = lcs / max(1, len(hypothesis))
    rec  = lcs / max(1, len(reference))
    if prec == 0 or rec == 0:
        return 0.0
    beta2 = beta * beta
    return ( (1 + beta2) * prec * rec ) / (rec + beta2 * prec)

In [ ]:
# -----------------------
# BLEU con suavizado
# -----------------------
_smooth = SmoothingFunction().method3

def bleu_scores(reference_tokens: List[str], hypothesis_tokens: List[str]) -> Tuple[float, float, float, float]:
    # BLEU-1..4 con pesos clásicos
    refs = [reference_tokens]  # NLTK espera lista de referencias
    try:
        b1 = sentence_bleu(refs, hypothesis_tokens, weights=(1,0,0,0), smoothing_function=_smooth)
        b2 = sentence_bleu(refs, hypothesis_tokens, weights=(0.5,0.5,0,0), smoothing_function=_smooth)
        b3 = sentence_bleu(refs, hypothesis_tokens, weights=(1/3,1/3,1/3,0), smoothing_function=_smooth)
        b4 = sentence_bleu(refs, hypothesis_tokens, weights=(0.25,0.25,0.25,0.25), smoothing_function=_smooth)
    except ZeroDivisionError:
        b1=b2=b3=b4=0.0
    return b1, b2, b3, b4

In [ ]:
# -----------------------
# METEOR
# -----------------------
def meteor(reference: str, hypothesis: str) -> float:
    # meteor_score requiere listas de referencias en bruto (strings)
    try:
        return float(meteor_score([reference], hypothesis))
    except Exception:
        # Fallback mínimo: F1 de unigramas (por si METEOR falla)
        ref_toks = set(tokenize(reference))
        hyp_toks = set(tokenize(hypothesis))
        if not ref_toks or not hyp_toks:
            return 0.0
        prec = len(ref_toks & hyp_toks) / len(hyp_toks)
        rec  = len(ref_toks & hyp_toks) / len(ref_toks)
        if prec+rec == 0:
            return 0.0
        return 2*prec*rec/(prec+rec)

In [ ]:
# -----------------------
# SPICE (opcional)
# -----------------------
_spice = None
def spice_score(reference: str, hypothesis: str) -> Optional[float]:
    """
    Calcula SPICE si pycocoevalcap está instalado (y Java disponible).
    Devuelve None si no está disponible.
    """
    global _spice
    if not _SPICE_AVAILABLE:
        return None
    if _spice is None:
        try:
            _spice = SPICE()
        except Exception:
            return None
    # SPICE espera dicts COCO-like: {imgId: ['ref sentence', ...]}
    # Aquí hacemos evaluación por-par. SPICE devuelve una lista de dicts con 'SPICE'
    try:
        refs = {0: [reference]}
        hyps = {0: [hypothesis]}
        out, _ = _spice.compute_score(refs, hyps)
        # out es lista con un dict por imagen; tomamos el primero
        return float(out[0]["SPICE"]["All"]["f"])
    except Exception:
        return None

In [ ]:
# -----------------------
# Proceso principal
# -----------------------
def evaluate_csv(in_path: str, out_path: str, include_spice: bool = True) -> pd.DataFrame:
    df = pd.read_csv(in_path)
    required = ["description_original", "description_image_captioning"]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Falta la columna requerida: '{col}'")

    results = []
    for idx, row in df.iterrows():
        ref_raw = row.get("description_original", "")
        hyp_raw = row.get("description_image_captioning", "")

        ref_tok = tokenize(ref_raw)
        hyp_tok = tokenize(hyp_raw)

        b1, b2, b3, b4 = bleu_scores(ref_tok, hyp_tok)
        rl = rouge_l(ref_tok, hyp_tok)
        met = meteor(ref_raw if isinstance(ref_raw, str) else "", hyp_raw if isinstance(hyp_raw, str) else "")

        entry = {
            "BLEU_1": b1,
            "BLEU_2": b2,
            "BLEU_3": b3,
            "BLEU_4": b4,
            "ROUGE_L": rl,
            "METEOR": met,
        }

        if include_spice:
            sc = spice_score(ref_raw if isinstance(ref_raw, str) else "", hyp_raw if isinstance(hyp_raw, str) else "")
            entry["SPICE"] = np.nan if sc is None else sc

        results.append(entry)

    metrics_df = pd.DataFrame(results)
    out_df = pd.concat([df.reset_index(drop=True), metrics_df], axis=1)
    out_df.to_csv(out_path, index=False)

    # Resumen en consola
    print("\n=== Resumen de métricas (promedio) ===")
    for col in ["BLEU_1","BLEU_2","BLEU_3","BLEU_4","ROUGE_L","METEOR","SPICE"]:
        if col in out_df.columns:
            vals = out_df[col].dropna().values
            if len(vals) > 0:
                print(f"{col:8s}: {np.mean(vals):.4f}")
            else:
                print(f"{col:8s}: N/A")

    if include_spice and "SPICE" in out_df.columns and out_df["SPICE"].isna().all():
        print("\nNota: SPICE no se calculó (pycocoevalcap/Java no disponible). "
              "Instala con: pip install git+https://github.com/salaniz/pycocoevalcap")

    print(f"\nArchivo con métricas guardado en: {out_path}")
    return out_df

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Evaluación de métricas de captioning sobre un CSV.")
    parser.add_argument("--in", dest="in_path", required=True, help="Ruta al CSV de entrada.")
    parser.add_argument("--out", dest="out_path", required=True, help="Ruta al CSV de salida.")
    parser.add_argument("--no-spice", action="store_true", help="Desactiva el cálculo de SPICE.")
    args = parser.parse_args()

    include_spice = not args.no_spice
    evaluate_csv(args.in_path, args.out_path, include_spice=include_spice)

In [ ]:
if __name__ == "__main__":
    main()